In [10]:
from pathlib import Path
import duckdb
import pandas as pd


DATA_DIR = Path("nyc 2025 data")
DB_PATH = "trips.duckdb"
ZONE_FILE = DATA_DIR / "taxi_zone_lookup.csv"


In [11]:
con = duckdb.connect(DB_PATH)

rows = []
yellow_total = 0
green_total = 0

for month in range(1, 13):
    mm = f"{month:02d}"

    yellow_file = DATA_DIR / f"yellow_tripdata_2025-{mm}.parquet"
    green_file = DATA_DIR / f"green_tripdata_2025-{mm}.parquet"

    yellow_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{yellow_file.as_posix()}')
    """).fetchone()[0]

    green_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{green_file.as_posix()}')
    """).fetchone()[0]

    yellow_total += yellow_count
    green_total += green_count

    rows.append({
        "month": f"2025-{mm}",
        "yellow_rows": yellow_count,
        "green_rows": green_count,
        "total_rows": yellow_count + green_count
    })

counts_df = pd.DataFrame(rows)

display(
    counts_df.style.format({
        "yellow_rows": "{:,}",
        "green_rows": "{:,}",
        "total_rows": "{:,}"
    })
)

print(f"Total Yellow rows for 2025: {yellow_total:,}")
print(f"Total Green rows for 2025:  {green_total:,}")
print(f"Grand Total rows for 2025:  {yellow_total + green_total:,}")

,month,yellow_rows,green_rows,total_rows
0,2025-01,"3,475,226","48,326","3,523,552"
1,2025-02,"3,577,543","46,621","3,624,164"
2,2025-03,"4,145,257","51,539","4,196,796"
3,2025-04,"3,970,553","52,132","4,022,685"
4,2025-05,"4,591,845","55,399","4,647,244"
5,2025-06,"4,322,960","49,390","4,372,350"
6,2025-07,"3,898,963","48,205","3,947,168"
7,2025-08,"3,574,091","46,306","3,620,397"
8,2025-09,"4,251,015","48,893","4,299,908"
9,2025-10,"4,428,699","49,416","4,478,115"


Total Yellow rows for 2025: 48,722,602
Total Green rows for 2025:  591,375
Grand Total rows for 2025:  49,313,977


In [12]:
yellow_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{(DATA_DIR / "yellow_tripdata_2025-*.parquet").as_posix()}',
        union_by_name = true
    )
""").df()

green_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{(DATA_DIR / "green_tripdata_2025-*.parquet").as_posix()}',
        union_by_name = true
    )
""").df()

yellow_schema = yellow_schema.rename(columns={
    "column_name": "column_name",
    "column_type": "yellow_type",
    "null": "yellow_null",
    "key": "yellow_key",
    "default": "yellow_default",
    "extra": "yellow_extra"
})

green_schema = green_schema.rename(columns={
    "column_name": "column_name",
    "column_type": "green_type",
    "null": "green_null",
    "key": "green_key",
    "default": "green_default",
    "extra": "green_extra"
})

schema_compare = yellow_schema.merge(
    green_schema,
    on="column_name",
    how="outer"
).sort_values("column_name").reset_index(drop=True)

display(schema_compare)

,column_name,yellow_type,yellow_null,yellow_key,yellow_default,yellow_extra,green_type,green_null,green_key,green_default,green_extra
0,Airport_fee,DOUBLE,YES,None,None,None,NaN,NaN,NaN,NaN,NaN
1,DOLocationID,INTEGER,YES,None,None,None,INTEGER,YES,None,None,None
2,PULocationID,INTEGER,YES,None,None,None,INTEGER,YES,None,None,None
3,RatecodeID,BIGINT,YES,None,None,None,BIGINT,YES,None,None,None
4,VendorID,INTEGER,YES,None,None,None,INTEGER,YES,None,None,None
5,cbd_congestion_fee,DOUBLE,YES,None,None,None,DOUBLE,YES,None,None,None
6,congestion_surcharge,DOUBLE,YES,None,None,None,DOUBLE,YES,None,None,None
7,ehail_fee,NaN,NaN,NaN,NaN,NaN,DOUBLE,YES,None,None,None
8,extra,DOUBLE,YES,None,None,None,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None,DOUBLE,YES,None,None,None


In [14]:
DATA_DIR = Path("nyc 2025 data")

con = duckdb.connect("trips.duckdb")

con.execute(f"""
    CREATE OR REPLACE TABLE raw_trips AS
    SELECT *
    FROM (
        SELECT *, 'yellow' AS service_type
        FROM read_parquet(
            '{(DATA_DIR / "yellow_tripdata_2025-*.parquet").as_posix()}',
            union_by_name = true
        )

        UNION ALL BY NAME

        SELECT *, 'green' AS service_type
        FROM read_parquet(
            '{(DATA_DIR / "green_tripdata_2025-*.parquet").as_posix()}',
            union_by_name = true
        )
    )
""")

con.execute("CHECKPOINT")

print("raw_trips created")

print(con.execute("""
    SELECT service_type, COUNT(*) AS trip_count
    FROM raw_trips
    GROUP BY service_type
    ORDER BY service_type
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw_trips created
  service_type  trip_count
0        green      591375
1       yellow    48722602


In [15]:

# Load zone lookup
con.execute(f"""
    CREATE OR REPLACE TABLE raw_zones AS
    SELECT *
    FROM read_csv_auto('{ZONE_FILE.as_posix()}')
""")

con.execute("CHECKPOINT")
print("raw_zones created")


In [16]:

# Verify
raw_trips_count = con.execute("SELECT COUNT(*) FROM raw_trips").fetchone()[0]
raw_zones_count = con.execute("SELECT COUNT(*) FROM raw_zones").fetchone()[0]

print(f"\nraw_trips loaded: {raw_trips_count:,} rows")
print(f"raw_zones loaded: {raw_zones_count:,} rows")

print("\nTrips by service type:")
print(con.execute("""
    SELECT service_type, COUNT(*) AS trip_count
    FROM raw_trips
    GROUP BY service_type
    ORDER BY service_type
""").df())

con.close()


raw_trips loaded: 49,313,977 rows
raw_zones loaded: 265 rows

Trips by service type:
  service_type  trip_count
0        green      591375
1       yellow    48722602
